# Test: GLM-4.7 (Zhipu AI)

Diversity agents — GPU 0,1,4,5 (TP=4 FP8, swaps with Primary), Port 8001

**Prerequisites:** vLLM server running on port 8001

**Note:** GLM-4.7 with `--reasoning-parser glm45` may put responses in
`reasoning_content` instead of `content`. Helper function below handles both.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from models.utils import GLM47

model = GLM47()
print('Model config:')
model.get_config()

def get_text(resp):
    """Extract text from GLM-4.7 response, handling reasoning parser output.
    
    GLM-4.7 with --reasoning-parser glm45 splits output into:
    - content: the final answer (may be None if model only reasons)
    - reasoning / reasoning_content: the chain-of-thought thinking
    """
    msg = resp.choices[0].message
    # Prefer content (the actual answer)
    if msg.content:
        return msg.content
    # Fall back to reasoning fields (vLLM uses 'reasoning' or 'reasoning_content')
    for field in ('reasoning', 'reasoning_content'):
        val = getattr(msg, field, None)
        if val:
            return val
    return '[empty response]'

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


## 1. Health Check

In [2]:
assert model.health_check(), 'Server not running on port 8001!'
print('Health check passed')

Health check passed


## 2. List Models

In [3]:
models = model.list_models()
for m in models:
    print(f'  {m.id}')

  /storage/data/AgenticCyOps_Private/models/zai-org/GLM-4.7-FP8


## 3. Chat Completions

In [4]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 3a. Basic chat
resp = model.chat(messages, max_tokens=100)
print('Basic chat:', get_text(resp))

Basic chat: 1.  **Analyze the Request:**
    *   **Role:** SOC Analyst (Security Operations Center Analyst).
    *   **Task:** Define "lateral movement attack".
    *   **Constraint:** Be concise.
    *   **Constraint:** One sentence.

2.  **Define "Lateral Movement":**
    *   What is it? It's the process by which an attacker moves through a network after gaining initial access.
    *   What is the goal? To access


In [5]:
# 3b. Deterministic (temp=0.0)
resp1 = model.chat_deterministic(messages, max_tokens=100)
resp2 = model.chat_deterministic(messages, max_tokens=100)
t1 = get_text(resp1)
t2 = get_text(resp2)
print('Deterministic r1:', t1[:80])
print('Deterministic r2:', t2[:80])
print('Match:', t1 == t2)

Deterministic r1: 1.  **Analyze the Request:**
    *   **Role:** SOC Analyst (Security Operations 
Deterministic r2: 1.  **Analyze the Request:**
    *   **Role:** SOC Analyst (Security Operations 
Match: True


In [6]:
# 3c. Creative (temp=0.7)
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', get_text(resp))

Creative: 1.  **Analyze the Request:**
    *   **Role:** SOC Analyst (Security Operations Center Analyst). This implies a focus on detection, defense, and technical accuracy, but also a need for clarity and brevity.
    *   **Topic:** Lateral movement attack.
    *   **Constraint:** One sentence.
    *   **Tone:** Concise.

2.  **Define "Lateral Movement":**
    *   What is it? It's the movement


In [7]:
# 3d. Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: 


## 4. Tool / Function Calling

GLM-4.7 uses `tool-call-parser glm47` and `reasoning-parser glm45` on vLLM side.

In [8]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'query_siem',
            'description': 'Search SIEM logs for security events',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Search query'},
                    'time_range': {'type': 'string', 'description': 'Time range'},
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'isolate_host',
            'description': 'Isolate a host from the network',
            'parameters': {
                'type': 'object',
                'properties': {
                    'hostname': {'type': 'string'},
                    'reason': {'type': 'string'},
                },
                'required': ['hostname', 'reason']
            }
        }
    }
]

In [9]:
# 4a. Auto tool choice
tc_messages = [
    {'role': 'system', 'content': 'You are a SOC analyst.'},
    {'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12 in the last hour.'}
]
resp = model.tool_call(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
if tc:
    print(f'Auto: {len(tc)} call(s)')
    for c in tc:
        print(f'  {c.function.name}({c.function.arguments})')
else:
    print(f'No tool calls. Response: {get_text(resp)[:100]}')

Auto: 1 call(s)
  query_siem({"query": "failed login AND source_ip=10.0.5.12", "time_range": "last hour"})


In [10]:
# 4b. Required tool choice
resp = model.tool_call_required(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
if tc:
    print(f'Required: {len(tc)} call(s)')
    for c in tc:
        print(f'  {c.function.name}({c.function.arguments})')
else:
    print(f'No tool calls. Response: {get_text(resp)[:100]}')

Required: 1 call(s)
  query_siem({"query": "failed login AND source_ip:10.0.5.12", "time_range": "last hour"})


In [11]:
# 4c. Specific tool choice
resp = model.tool_call_specific(tc_messages, tools, 'isolate_host')
tc = resp.choices[0].message.tool_calls
if tc:
    print(f'Specific (isolate_host): {len(tc)} call(s)')
    for c in tc:
        print(f'  {c.function.name}({c.function.arguments})')
else:
    print(f'No tool calls. Response: {get_text(resp)[:100]}')

Specific (isolate_host): 1 call(s)
  isolate_host({
  "hostname": "10.0.5.12",
  "reason": "Investigating failed login attempts"
})


## 5. Structured Output

In [12]:
# 5a. JSON mode
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Multiple failed SSH logins from 10.0.5.12". Return {"severity": str, "category": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', get_text(resp))

JSON mode: 1.  **Analyze the Request:**
    *   Input string: "Multiple failed SSH logins from 10.0.5.12".
    *   Output format: JSON only.
    *   Required fields: `severity` (str), `category` (str), `confidence` (float).

2.  **Analyze the Input String:**
    *   "Multiple failed SSH logins": This indicates a brute-force attack or a password-guessing attempt against the SSH service. It's a security event.
    *   "from 10.0.5.12": This identifies the source IP address.

3.  **Determine `severity`:**
    *   Failed logins are common, but "multiple" suggests an automated attack.
    *   It's not a confirmed breach (successful login), so it's not "Critical".
    *   It's not just a single typo, so it's not "Low" or "Info


In [13]:
# 5b. JSON schema
schema = {
    'type': 'object',
    'properties': {
        'severity': {'type': 'string', 'enum': ['low', 'medium', 'high', 'critical']},
        'category': {'type': 'string'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1}
    },
    'required': ['severity', 'category', 'confidence']
}
resp = model.chat_json_schema(json_messages, schema, temperature=0.0, max_tokens=200)
print('JSON schema:', get_text(resp))

JSON schema: 1.  **Analyze the Request:**
    *   Input string: "Multiple failed SSH logins from 10.0.5.12".
    *   Output format: JSON only.
    *   Required fields: `severity` (str), `category` (str), `confidence` (float).

2.  **Analyze the Input String:**
    *   "Multiple failed SSH logins": This indicates a brute-force attack or a password-guessing attempt against the SSH service. It's a security event.
    *   "from 10.0.5.12": This identifies the source IP address.

3.  **Determine `severity`:**
    *   Failed logins are common, but "multiple" suggests an automated attack.
    *   It's not a confirmed breach (successful login), so it's not "Critical".
    *   It's not just a single typo, so it's not "Low" or "Info


## 6. Batch Inference

In [14]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
    [{'role': 'user', 'content': 'What is a zero-day? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {get_text(r)}')

Batch 0: 1.  **Analyze the Request:**
    *   **Topic:** Phishing.
    *   **Constraint:** One sentence.

2.  **Define "Phishing":**
    *   It's a type of cybercrime.
    *   It involves deception (social engineering).
    *   The goal is to steal sensitive data (passwords, credit card numbers) or install malware
Batch 1: 1.  **Analyze the Request:**
    *   **Topic:** Ransomware.
    *   **Constraint:** One sentence.

2.  **Define Ransomware:**
    *   What is it? Malicious software (malware).
    *   What does it do? Encrypts files or locks a user out of their system.
    *   What is the goal
Batch 2: 1.  **Analyze the Request:**
    *   **Topic:** Zero-day (vulnerability/exploit).
    *   **Constraint:** One sentence.

2.  **Define "Zero-day":**
    *   What is it? A software security flaw.
    *   What makes it special? It is unknown to the vendor/developer.
    *   What is the consequence


## 7. Token Usage

In [15]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     25
Completion tokens: 100
Total tokens:      125


## 8. GLM-4.7 vs Qwen3-235B Comparison

Quick check that GLM-4.7 produces different outputs (model diversity for Eval A).

In [16]:
# Compare same prompt on both models (requires port 8000 running too)
from models.utils import Qwen3_235B

qwen = Qwen3_235B()
test_msg = [{'role': 'user', 'content': 'Classify this alert as HIGH/MEDIUM/LOW: "Unusual outbound DNS traffic to known C2 domain"'}]

if qwen.health_check():
    r_qwen = qwen.chat_deterministic(test_msg, max_tokens=50)
    r_glm = model.chat_deterministic(test_msg, max_tokens=50)
    print(f'Qwen3-235B: {r_qwen.choices[0].message.content}')
    print(f'GLM-4.7:    {get_text(r_glm)}')
else:
    print('Qwen3-235B server not running, skipping comparison')

Qwen3-235B server not running, skipping comparison


## 9. Get Config

In [17]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "zai-org/GLM-4.7",
  "model_path": "/storage/data/AgenticCyOps_Private/models/zai-org/GLM-4.7-FP8",
  "role": "diversity_agents",
  "architecture": "dense",
  "gpu_assignment": "0,1,4,5",
  "port": 8001,
  "base_url": "http://localhost:8001/v1",
  "tool_call_parser": "glm47",
  "reasoning_parser": "glm45",
  "requires_vllm_nightly": true
}


## Summary

All tests passed if no cells raised exceptions above.